# Attention latent injection run

该 Notebook 用于 Colab 冷启动: 挂载 Google Drive、拉取仓库、加载 SD3.5 Medium、读取真实 attention geometry 包、重建 attention-relative carrier, 在真实 latent callback 中执行 attention-relative update, 生成 paired images、真实 latent update records 和质量指标, 并把所有核对文件打包到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/attention_latent_injection.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 SD3.5 Medium 访问权限, 并配置 `HF_TOKEN`。
3. 确认 Drive 中已有 attention geometry 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `attention_geometry/attention_geometry_package_*.zip``。
4. 默认输出目录为 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `attention_latent_injection``。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
paper_run_name = SLM_WM_PAPER_RUN_NAME.strip() or "pilot_paper"
os.environ['SLM_WM_PAPER_RUN_NAME'] = paper_run_name
prompt_file_by_run = {
    'pilot_paper': 'configs/paper_main_pilot_paper_prompts.txt',
    'full_paper': 'configs/paper_main_full_paper_prompts.txt',
}
drive_result_root = f'/content/drive/MyDrive/SLM/{paper_run_name}_results'
os.environ['SLM_WM_DRIVE_RESULT_ROOT'] = drive_result_root
paper_run_sample_count = os.environ.get('SLM_WM_PAPER_RUN_SAMPLE_COUNT', 'all')
os.environ['SLM_WM_PAPER_RUN_SAMPLE_COUNT'] = paper_run_sample_count
prompt_count_by_run = {'pilot_paper': 600, 'full_paper': 6000}
def resolve_paper_run_count(value):
    normalized = str(value).strip().lower()
    if normalized in {'', 'all', 'none', 'unlimited'}:
        return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper']))
    resolved = int(normalized)
    return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper'])) if resolved <= 0 else resolved
paper_run_expected_sample_count = resolve_paper_run_count(paper_run_sample_count)
paper_run_minimum_clean_negative_count = os.environ.get('SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT'] = paper_run_minimum_clean_negative_count
paper_run_dataset_minimum_count = os.environ.get('SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT'] = paper_run_dataset_minimum_count
target_fpr_by_run = {'pilot_paper': '0.01', 'full_paper': '0.001'}
paper_run_target_fpr = os.environ.get('SLM_WM_PAPER_RUN_TARGET_FPR', target_fpr_by_run.get(paper_run_name, target_fpr_by_run['pilot_paper']))
os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'] = paper_run_target_fpr
paper_run_target_fpr_token = paper_run_target_fpr.replace('.', '_')
os.environ['SLM_WM_PROTOCOL_PROFILE'] = f'{paper_run_name}_fixed_fpr_{paper_run_target_fpr_token}'
os.environ['SLM_WM_PROMPT_SET'] = paper_run_name
os.environ['SLM_WM_PROMPT_FILE'] = prompt_file_by_run.get(paper_run_name, prompt_file_by_run['pilot_paper'])
os.environ['SLM_WM_DRIVE_OUTPUT_DIR'] = f'{drive_result_root}/attention_latent_injection'
os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'] = f'{drive_result_root}/attention_geometry'
os.environ['SLM_WM_ATTENTION_SUBSPACE_RECORDS'] = paper_run_sample_count
os.environ.setdefault('SLM_WM_ATTENTION_RUNTIME_STRENGTH', '0.025')
print({
    'protocol_profile': os.environ['SLM_WM_PROTOCOL_PROFILE'],
    'prompt_set': os.environ['SLM_WM_PROMPT_SET'],
    'prompt_file': os.environ['SLM_WM_PROMPT_FILE'],
    'drive_output_dir': os.environ['SLM_WM_DRIVE_OUTPUT_DIR'],
    'attention_geometry_drive_dir': os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'],
    'attention_subspace_records': os.environ['SLM_WM_ATTENTION_SUBSPACE_RECORDS'],
    'attention_runtime_strength': os.environ['SLM_WM_ATTENTION_RUNTIME_STRENGTH'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
import importlib
import importlib.metadata as importlib_metadata
import importlib.util

# 只做只读依赖诊断, 不清理 sys.modules, 避免在同一 Colab 内核中重载 numpy / torch。
importlib.invalidate_caches()

import diffusers
import transformers
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPTextModelWithProjection
from transformers.generation import GenerationMixin

dependency_report = {
    'diffusers': diffusers.__version__,
    'transformers': transformers.__version__,
    'numpy': importlib_metadata.version('numpy'),
    'tokenizers': importlib_metadata.version('tokenizers'),
    'accelerate': importlib_metadata.version('accelerate'),
    'huggingface_hub': importlib_metadata.version('huggingface_hub'),
    'scipy_available': importlib.util.find_spec('scipy') is not None,
    'torchvision_available': importlib.util.find_spec('torchvision') is not None,
    'stable_diffusion_pipeline': StableDiffusion3Pipeline.__name__,
    'clip_text_model_with_projection': CLIPTextModelWithProjection.__name__,
    'generation_mixin': GenerationMixin.__name__,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from pathlib import Path

geometry_dir = Path(os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'])
geometry_packages = sorted(geometry_dir.glob('attention_geometry_package_*.zip'))
assert geometry_packages, f'未找到 attention geometry 包: {geometry_dir}'
print({'geometry_package': str(geometry_packages[-1])})


In [ ]:
from paper_workflow.colab_utils.attention_latent_injection import run_default_attention_latent_injection_plan

attention_injection_summary = run_default_attention_latent_injection_plan(root='.')
assert attention_injection_summary['run_decision'] == 'pass', attention_injection_summary
assert attention_injection_summary['latent_update_count'] > 0, attention_injection_summary
assert attention_injection_summary['image_quality_metrics_ready'] is True, attention_injection_summary
attention_injection_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.attention_latent_injection import package_attention_latent_injection_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_DRIVE_OUTPUT_DIR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'attention_latent_injection_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_attention_latent_injection_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/attention_latent_injection').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/attention_latent_update').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))
